# MBTI Classifier — NLP Text Classification Notebook

Notebook ini adalah versi portfolio-ready yang lebih bersih dari project MBTI Classifier.

Fokus notebook:
- membaca dataset dari `data/dataset.zip`
- melakukan text preprocessing
- membuat fitur bag-of-words sederhana
- melatih model Naive Bayes
- mengevaluasi model
- membuat fungsi prediksi MBTI-style dari input teks anonim

> Disclaimer: Project ini hanya untuk demo edukasi/portfolio. Hasil prediksi bukan tes psikologis resmi.

## 1. Import Libraries

In [ ]:
import os
import re
import zipfile
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import stopwords

## 2. Optional NLTK Resource Setup

Jika stopwords belum tersedia, jalankan download otomatis.  
Bagian ini aman untuk local notebook. Untuk Streamlit deployment, app bisa memakai preprocessing regex-based agar lebih stabil.

In [ ]:
try:
    STOPWORDS = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    STOPWORDS = set(stopwords.words("english"))

## 3. Load Dataset from `data/dataset.zip`

Notebook ini sengaja memakai path:

```text
data/dataset.zip
```

Pastikan struktur folder lokal kamu seperti ini:

```text
mbti-classifier-nlp/
├── data/
│   └── dataset.zip
└── mbti_classifier_notebook.ipynb
```

In [ ]:
DATASET_ZIP_PATH = "data/dataset.zip"

if not os.path.exists(DATASET_ZIP_PATH):
    raise FileNotFoundError(
        "Dataset tidak ditemukan. Pastikan file berada di: data/dataset.zip"
    )

with zipfile.ZipFile(DATASET_ZIP_PATH, "r") as z:
    csv_files = [name for name in z.namelist() if name.lower().endswith(".csv")]
    if not csv_files:
        raise FileNotFoundError("Tidak ada file CSV di dalam data/dataset.zip")

    csv_name = csv_files[0]
    with z.open(csv_name) as f:
        data_set = pd.read_csv(f)

print("CSV loaded from:", csv_name)
print("Dataset shape:", data_set.shape)
data_set.head()

## 4. Basic Dataset Inspection

In [ ]:
print("Columns:", data_set.columns.tolist())
print("\nMissing values:")
print(data_set.isnull().sum())

print("\nUnique MBTI types:")
print(sorted(data_set["type"].unique()))

print("\nClass distribution:")
print(data_set["type"].value_counts())

In [ ]:
plt.figure(figsize=(10, 5))
data_set["type"].value_counts().plot(kind="bar")
plt.title("MBTI Type Distribution")
plt.xlabel("MBTI Type")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Text Preprocessing

Tahapan preprocessing:
- lowercase
- hapus URL
- hapus delimiter `|||`
- hapus karakter non-huruf
- tokenisasi sederhana berbasis regex
- stopword removal
- hapus token terlalu pendek

Preprocessing dibuat sederhana agar cocok dengan baseline Naive Bayes.

In [ ]:
def preprocess_text(text):
    if pd.isna(text):
        return []

    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = text.replace("|||", " ")
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()

    tokens = [
        token for token in tokens
        if token not in STOPWORDS and len(token) > 2
    ]

    return tokens


sample_tokens = preprocess_text(data_set.loc[0, "posts"])
sample_tokens[:30]

## 6. Prepare Data

Dataset asli memiliki kolom:
- `type` sebagai label MBTI
- `posts` sebagai teks input

Untuk menjaga runtime notebook tetap wajar, kamu bisa membatasi jumlah data dengan `MAX_ROWS`.  
Jika ingin pakai semua data, set `MAX_ROWS = None`.

In [ ]:
RANDOM_STATE = 42
MAX_ROWS = None  # ubah ke angka seperti 3000 jika laptop lambat

df = data_set[["type", "posts"]].dropna().copy()

if MAX_ROWS is not None and len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=RANDOM_STATE).reset_index(drop=True)

df["tokens"] = df["posts"].apply(preprocess_text)

print("Prepared data shape:", df.shape)
df[["type", "tokens"]].head()

## 7. Train-Test Split

Split dilakukan sederhana tanpa scikit-learn agar dependency tetap ringan dan sesuai baseline project.

In [ ]:
def train_test_split_dataframe(dataframe, test_size=0.2, random_state=42):
    shuffled = dataframe.sample(frac=1, random_state=random_state).reset_index(drop=True)
    split_index = int(len(shuffled) * (1 - test_size))
    train_df = shuffled.iloc[:split_index].reset_index(drop=True)
    test_df = shuffled.iloc[split_index:].reset_index(drop=True)
    return train_df, test_df


train_df, test_df = train_test_split_dataframe(df, test_size=0.2, random_state=RANDOM_STATE)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

## 8. Build Vocabulary and Bag-of-Words Features

Kita ambil kata paling sering dari training set sebagai vocabulary.  
Fitur dibuat sebagai binary bag-of-words: apakah sebuah kata muncul dalam teks atau tidak.

In [ ]:
MAX_FEATURES = 3000

word_counts = Counter()
for tokens in train_df["tokens"]:
    word_counts.update(tokens)

VOCAB = [word for word, count in word_counts.most_common(MAX_FEATURES)]

print("Vocabulary size:", len(VOCAB))
print("Top 20 words:", VOCAB[:20])

In [ ]:
def document_features(tokens, vocab=VOCAB):
    token_set = set(tokens)
    return {f"contains({word})": (word in token_set) for word in vocab}


example_features = document_features(train_df.loc[0, "tokens"])
list(example_features.items())[:10]

## 9. Baseline 16-Class MBTI Classifier

Model ini langsung memprediksi 16 tipe MBTI.  
Biasanya pendekatan ini lebih sulit karena jumlah kelas lebih banyak dan distribusi kelas tidak seimbang.

In [ ]:
train_features_16 = [
    (document_features(tokens), label)
    for tokens, label in zip(train_df["tokens"], train_df["type"])
]

test_features_16 = [
    (document_features(tokens), label)
    for tokens, label in zip(test_df["tokens"], test_df["type"])
]

classifier_16 = NaiveBayesClassifier.train(train_features_16)

train_accuracy_16 = nltk.classify.accuracy(classifier_16, train_features_16)
test_accuracy_16 = nltk.classify.accuracy(classifier_16, test_features_16)

print("16-Class MBTI Classifier")
print("Train Accuracy:", round(train_accuracy_16, 4))
print("Test Accuracy :", round(test_accuracy_16, 4))

## 10. Trait-Based Classifiers

Untuk demo recruiter, pendekatan yang lebih mudah dijelaskan adalah memecah MBTI menjadi 4 classifier biner:

1. I vs E
2. N vs S
3. T vs F
4. J vs P

Output akhirnya digabung menjadi satu tipe MBTI-style.

In [ ]:
TRAITS = {
    "IE": ("I", "E"),
    "NS": ("N", "S"),
    "TF": ("T", "F"),
    "JP": ("J", "P"),
}


def get_trait_label(mbti_type, trait_pair):
    left, right = trait_pair
    if left in mbti_type:
        return left
    if right in mbti_type:
        return right
    return None


def build_trait_dataset(dataframe, trait_pair):
    items = []
    for tokens, mbti_type in zip(dataframe["tokens"], dataframe["type"]):
        label = get_trait_label(mbti_type, trait_pair)
        if label is not None:
            items.append((document_features(tokens), label))
    return items

In [ ]:
trait_classifiers = {}
trait_results = []

for trait_name, trait_pair in TRAITS.items():
    train_features = build_trait_dataset(train_df, trait_pair)
    test_features = build_trait_dataset(test_df, trait_pair)

    clf = NaiveBayesClassifier.train(train_features)

    train_acc = nltk.classify.accuracy(clf, train_features)
    test_acc = nltk.classify.accuracy(clf, test_features)

    trait_classifiers[trait_name] = clf
    trait_results.append({
        "trait": trait_name,
        "labels": " vs ".join(trait_pair),
        "train_accuracy": train_acc,
        "test_accuracy": test_acc
    })

trait_results_df = pd.DataFrame(trait_results)
trait_results_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(trait_results_df["labels"], trait_results_df["test_accuracy"])
plt.title("Trait Classifier Test Accuracy")
plt.xlabel("Trait Classifier")
plt.ylabel("Test Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 11. Prediction Function

Fungsi berikut menerima input teks anonim, melakukan preprocessing, lalu memprediksi MBTI-style result.  
Confidence ditampilkan jika tersedia dari `prob_classify`.

In [ ]:
def predict_mbti(text):
    tokens = preprocess_text(text)
    features = document_features(tokens)

    predicted_letters = []
    confidence_details = {}

    for trait_name, trait_pair in TRAITS.items():
        clf = trait_classifiers[trait_name]
        prob_dist = clf.prob_classify(features)
        predicted_label = prob_dist.max()
        confidence = prob_dist.prob(predicted_label)

        predicted_letters.append(predicted_label)
        confidence_details[trait_name] = {
            "prediction": predicted_label,
            "confidence": confidence,
            "labels": trait_pair
        }

    mbti_prediction = "".join(predicted_letters)

    return {
        "mbti_prediction": mbti_prediction,
        "confidence_details": confidence_details,
        "tokens": tokens
    }

## 12. Anonymous Demo Prediction

Bagian ini menggantikan contoh lama yang memakai file personal.  
Untuk portfolio public, gunakan sample anonim seperti ini.

In [ ]:
anonymous_sample_text = '''
I enjoy exploring abstract ideas, reflecting on future possibilities,
and writing about personal growth. I usually prefer meaningful conversations,
structured plans, and learning independently.
'''

prediction_result = predict_mbti(anonymous_sample_text)

print("Predicted MBTI-style type:", prediction_result["mbti_prediction"])
print("\nTrait confidence:")
for trait, detail in prediction_result["confidence_details"].items():
    print(
        trait,
        "=>",
        detail["prediction"],
        "| confidence:",
        round(detail["confidence"], 4)
    )

print("\nProcessed tokens sample:")
print(prediction_result["tokens"][:30])

## 13. Important Disclaimer

Project ini adalah demo edukasi NLP dan machine learning.  
Prediksi yang dihasilkan bukan hasil psikologi resmi, bukan diagnosis, dan tidak boleh digunakan untuk hiring, counseling, medical decision, atau keputusan sensitif lain.

Nilai utama project ini adalah:
- memahami workflow NLP,
- preprocessing teks,
- baseline classification,
- model evaluation,
- deployment demo sederhana.

## 14. Suggested Next Improvements

Beberapa pengembangan lanjutan:
- gunakan TF-IDF,
- coba Logistic Regression atau Linear SVM,
- gunakan stratified train-test split,
- tambahkan classification report,
- simpan model dengan pickle/joblib,
- hubungkan model tersimpan ke Streamlit,
- buat model card dan limitation section.